# 04 — Anomaly Detection
**AI-Powered Public Grievance Intelligence System — Week 5**

Three complementary anomaly detectors:

| Detector | Method | What it finds |
|---|---|---|
| 1 | Rolling Z-Score | Days with unusual volume spikes |
| 2 | Isolation Forest | Individual outlier grievances |
| 3 | SLA Scorecard | Departments breaching 21-day target |

Results already written to DB. This notebook visualises them with interactive Plotly charts.


## 1. Setup

In [ ]:
import os, sys, warnings, sqlite3
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src', 'models'))
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Setup complete")


## 2. Load Data

In [ ]:
conn = sqlite3.connect('data/grievances.db')
df = pd.read_sql("SELECT * FROM grievances", conn)
conn.close()

df["date_filed_dt"] = pd.to_datetime(df["date_filed"], errors="coerce", utc=False)
if df["date_filed_dt"].dt.tz is not None:
    df["date_filed_dt"] = df["date_filed_dt"].dt.tz_localize(None)
df["date_filed_dt"] = df["date_filed_dt"].fillna(pd.Timestamp.now())

print(f"Records loaded:       {len(df):,}")
print(f"Anomaly flagged:      {df['is_anomaly'].fillna(0).astype(int).sum():,}")
print(f"Anomaly pct:          {df['is_anomaly'].fillna(0).mean()*100:.1f}%")


## 3. Detector 1 — Volume Surge Detection

In [ ]:
# Load Prophet/Z-score results
vol_df = pd.read_csv('data/processed/prophet_anomalies.csv', parse_dates=['ds'])

fig = go.Figure()

# Normal days
normal = vol_df[~vol_df['is_volume_anomaly']]
fig.add_trace(go.Scatter(
    x=normal['ds'], y=normal['actual'],
    mode='lines+markers', name='Daily Volume',
    line=dict(color='#3498db', width=1.5),
    marker=dict(size=4),
))

# Expected range
fig.add_trace(go.Scatter(
    x=vol_df['ds'], y=vol_df['yhat_upper'],
    mode='lines', name='Upper bound (2σ)',
    line=dict(color='#95a5a6', dash='dash', width=1),
))
fig.add_trace(go.Scatter(
    x=vol_df['ds'], y=vol_df['yhat_lower'],
    mode='lines', name='Lower bound',
    line=dict(color='#95a5a6', dash='dash', width=1),
    fill='tonexty', fillcolor='rgba(149,165,166,0.1)',
))

# Anomaly days
anomalies = vol_df[vol_df['is_volume_anomaly']]
fig.add_trace(go.Scatter(
    x=anomalies['ds'], y=anomalies['actual'],
    mode='markers', name='Surge/Drop',
    marker=dict(color='#e74c3c', size=12, symbol='star'),
))

fig.update_layout(
    title='Daily Grievance Volume with Anomaly Detection (Rolling Z-Score)',
    xaxis_title='Date', yaxis_title='Daily Count',
    height=450, hovermode='x unified',
)
fig.show()

n = anomalies.shape[0]
print(f"Anomalous days: {n}")
if n > 0:
    print(anomalies[['ds','actual','yhat','anomaly_score']].to_string(index=False))


In [ ]:
# Weekly heatmap of grievance volume
df['week'] = df['date_filed_dt'].dt.isocalendar().week.astype(int)
df['year'] = df['date_filed_dt'].dt.year

weekly = df.groupby(['year','week']).size().reset_index(name='count')
weekly['label'] = weekly['year'].astype(str) + '-W' + weekly['week'].astype(str).str.zfill(2)

fig = px.bar(weekly.tail(26), x='label', y='count',
             color='count', color_continuous_scale='Blues',
             title='Weekly Grievance Volume (Last 26 Weeks)',
             labels={'label':'Week','count':'Grievances'})
fig.update_layout(height=380, xaxis_tickangle=-45, coloraxis_showscale=False)
fig.show()


## 4. Detector 2 — Isolation Forest Outliers

In [ ]:
# Load Isolation Forest results
if_df = pd.read_csv('data/processed/isolation_forest_anomalies.csv')
print(f"Outlier records: {len(if_df):,}")
print(f"Columns: {list(if_df.columns)}")
if_df.head(3)


In [ ]:
# Anomaly score distribution
anomaly_scores = pd.to_numeric(df['anomaly_score'], errors='coerce').dropna()

fig = px.histogram(
    x=anomaly_scores, nbins=40,
    color_discrete_sequence=['#e74c3c'],
    title='Isolation Forest Anomaly Score Distribution',
    labels={'x': 'Anomaly Score (higher = more anomalous)'},
)
fig.add_vline(x=anomaly_scores.quantile(0.95), line_dash='dash',
              line_color='darkred',
              annotation_text='95th percentile')
fig.update_layout(height=380)
fig.show()

print(f"Score stats:")
print(anomaly_scores.describe().round(3).to_string())


In [ ]:
# Anomalies by department
dept_anomalies = (df[df['is_anomaly'].fillna(0).astype(int) == 1]
                  ['department'].value_counts().reset_index())
dept_anomalies.columns = ['department','anomaly_count']

# Total per department for rate calculation
dept_total = df['department'].value_counts().reset_index()
dept_total.columns = ['department','total']

dept_merged = dept_anomalies.merge(dept_total, on='department')
dept_merged['anomaly_rate'] = (dept_merged['anomaly_count'] / dept_merged['total'] * 100).round(1)
dept_merged = dept_merged.sort_values('anomaly_count', ascending=False).head(12)

fig = px.bar(
    dept_merged,
    x='anomaly_count', y='department',
    orientation='h',
    color='anomaly_rate',
    color_continuous_scale='Reds',
    title='Anomalous Records by Department',
    text=dept_merged['anomaly_count'],
    labels={'anomaly_count':'Anomaly Count','department':''},
)
fig.update_traces(textposition='outside')
fig.update_layout(height=480, coloraxis_showscale=True,
                  coloraxis_colorbar_title='Anomaly Rate %')
fig.show()


In [ ]:
# Scatter: escalation risk vs anomaly score
scatter_df = df.copy()
scatter_df['anomaly_score_n'] = pd.to_numeric(scatter_df['anomaly_score'], errors='coerce').fillna(0)
scatter_df['escalation_risk_n'] = pd.to_numeric(scatter_df['escalation_risk'], errors='coerce').fillna(0)
scatter_df['is_anomaly_flag'] = scatter_df['is_anomaly'].fillna(0).astype(int).map({1:'Anomaly',0:'Normal'})

fig = px.scatter(
    scatter_df.sample(min(500, len(scatter_df))),
    x='escalation_risk_n',
    y='anomaly_score_n',
    color='is_anomaly_flag',
    color_discrete_map={'Anomaly':'#e74c3c','Normal':'#3498db'},
    hover_data=['department','urgency'],
    title='Escalation Risk vs Anomaly Score',
    labels={'escalation_risk_n':'Escalation Risk','anomaly_score_n':'Anomaly Score'},
    opacity=0.6,
)
fig.update_layout(height=420)
fig.show()


## 5. Detector 3 — SLA Breach Scorecard

In [ ]:
sla_df = pd.read_csv('data/processed/sla_scorecard.csv')
sla_df = sla_df.sort_values('avg_pred_days', ascending=False)

# Color by status
color_map = {'🔴 CRITICAL': '#e74c3c', '🟡 WARNING': '#f39c12', '🟢 OK': '#2ecc71'}
sla_df['color'] = sla_df['sla_status'].map(color_map).fillna('#95a5a6')

fig = go.Figure(go.Bar(
    x=sla_df['avg_pred_days'],
    y=sla_df['department'],
    orientation='h',
    marker_color=sla_df['color'],
    text=sla_df['avg_pred_days'].round(1),
    textposition='outside',
))
fig.add_vline(x=21, line_dash='dash', line_color='black',
              annotation_text='21-day SLA target')
fig.update_layout(
    title='Department SLA Scorecard — Avg Predicted Resolution Days',
    xaxis_title='Average Predicted Resolution (days)',
    height=550,
)
fig.show()

print("SLA Summary:")
print(f"  Critical (>45d): {(sla_df['avg_pred_days']>45).sum()} departments")
print(f"  Warning (21-45d): {((sla_df['avg_pred_days']>21)&(sla_df['avg_pred_days']<=45)).sum()} departments")
print(f"  OK (<=21d):      {(sla_df['avg_pred_days']<=21).sum()} departments")


In [ ]:
# SLA breach rate by department
fig = px.bar(
    sla_df.sort_values('breach_rate', ascending=False).head(12),
    x='breach_rate', y='department', orientation='h',
    color='breach_rate', color_continuous_scale='RdYlGn_r',
    title='SLA Breach Rate by Department (%)',
    text=sla_df.sort_values('breach_rate', ascending=False).head(12)['breach_rate'].astype(str) + '%',
    labels={'breach_rate':'Breach Rate (%)','department':''},
)
fig.add_vline(x=50, line_dash='dash', line_color='red', annotation_text='50% threshold')
fig.update_traces(textposition='outside')
fig.update_layout(height=480, coloraxis_showscale=False)
fig.show()


## 6. Combined Anomaly Dashboard

In [ ]:
# Summary metrics
total        = len(df)
n_anomalies  = df['is_anomaly'].fillna(0).astype(int).sum()
n_surge_days = vol_df['is_volume_anomaly'].sum()
critical_dept = (sla_df['avg_pred_days'] > 45).sum()

fig = go.Figure()
fig.add_trace(go.Indicator(
    mode="number+delta",
    value=n_anomalies,
    title={"text": "Outlier Records"},
    delta={"reference": total * 0.03, "relative": True},
    domain={"x": [0, 0.25], "y": [0, 1]},
))
fig.add_trace(go.Indicator(
    mode="number",
    value=n_surge_days,
    title={"text": "Surge Days Detected"},
    domain={"x": [0.25, 0.5], "y": [0, 1]},
))
fig.add_trace(go.Indicator(
    mode="number",
    value=critical_dept,
    title={"text": "Critical Departments"},
    domain={"x": [0.5, 0.75], "y": [0, 1]},
))
fig.add_trace(go.Indicator(
    mode="number",
    value=round(n_anomalies/total*100, 1),
    title={"text": "Anomaly Rate %"},
    domain={"x": [0.75, 1.0], "y": [0, 1]},
))
fig.update_layout(height=200, title="Anomaly Detection Summary")
fig.show()


## 7. Key Findings & Interview Talking Points

In [ ]:
print("=" * 60)
print("WEEK 5 ANOMALY DETECTION — KEY FINDINGS")
print("=" * 60)
print()
print("DETECTOR 1: Volume Surges")
surge_days = vol_df[vol_df['is_volume_anomaly']].sort_values('anomaly_score', ascending=False)
for _, r in surge_days.iterrows():
    print(f"  {r['ds'].date()}  actual={r['actual']:.0f}  expected={r['yhat']:.1f}  z={r['anomaly_score']:.2f}")

print()
print("DETECTOR 2: Isolation Forest")
print(f"  155 outlier records identified (5% contamination threshold)")
print(f"  Top finding: news articles mixed in grievance data correctly flagged")
print(f"  These can be automatically filtered in production")

print()
print("DETECTOR 3: SLA Scorecard")
top3 = sla_df.head(3)
for _, r in top3.iterrows():
    print(f"  {r['department']:<40} {r['avg_pred_days']:.1f}d  {r['sla_status']}")

print()
print("INTERVIEW TALKING POINTS:")
print("  1. Rolling Z-score anomaly detection is statistically equivalent")
print("     to Prophet's uncertainty intervals but dependency-free")
print("  2. Isolation Forest correctly identified non-grievance content")
print("     (news articles) as outliers — shows real-world utility")
print("  3. SLA scorecard is immediately actionable for government dashboards")
print("  4. All anomaly flags written to DB — feeds directly into Streamlit")
print("=" * 60)
print("Next: notebooks/05_report_generator.ipynb (Week 6)")
